# Triagem por título e resumo



## 1) Instalação

In [ ]:
!pip -q install -U openai openpyxl tenacity tqdm

## 2) Imports e configuração


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import re
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from openai import OpenAI, APITimeoutError, APIConnectionError, RateLimitError, APIError

# ========= AJUSTE ESTES CAMINHOS =========
INPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/TCC/lista_final_artigos_deduplicada_com_abstract.csv"
OUTPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/TCC/screened_artigos.csv"

# ========= MODELO =========
# Troque se necessário pelo modelo disponível na sua conta
MODEL = "gpt-5.4"

# ========= CONTROLE =========
REQUEST_PAUSE_SECONDS = 0.8
PROMPT_VERSION = "js_screening_v2"
SAVE_EVERY_N = 10


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3) API KEY


In [ ]:
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY não encontrada em Colab Secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)


## 4) Critérios atuais do seu protocolo


In [ ]:
TOPIC = (
    "Systematic mapping study on tools and approaches for detecting code smells in JavaScript."
)

INCLUSION_CRITERIA = [
    "Primary study published in a peer-reviewed venue.",
    "Main focus is JavaScript.",
    "Addresses code smells or equivalent terms such as bad smell, design smell, anti-pattern, or antipattern.",
    "Presents, uses, evaluates, or compares a tool, technique, metric, or approach for detecting code smells.",
    "Title and abstract provide enough information to judge relevance."
]

EXCLUSION_CRITERIA = [
    "Secondary study, such as a systematic review, mapping study, or survey.",
    "Grey literature, such as thesis, dissertation, white paper, blog post, editorial, tutorial, or extended abstract.",
    "Main focus is not JavaScript.",
    "Does not address code smells or equivalent terms.",
    "Does not present, use, evaluate, or compare any detection tool, technique, metric, or approach.",
]

## 5) Prompt


In [ ]:
def build_prompt(title: str, abstract: str, year: str = "", source: str = "", doi: str = "", conclusion: str = "") -> str:
    inc = "\n".join([f"- {x}" for x in INCLUSION_CRITERIA])
    exc = "\n".join([f"- {x}" for x in EXCLUSION_CRITERIA])

    return f"""
You are assisting with title-and-abstract screening for a systematic mapping study in Software Engineering.

TOPIC:
{TOPIC}

INCLUSION CRITERIA:
{inc}

EXCLUSION CRITERIA:
{exc}

SCREENING INSTRUCTIONS:
1. Use only the information provided below.
2. If the paper clearly matches the topic, return INCLUDE.
3. If the paper clearly violates one or more exclusion criteria, return EXCLUDE.
4. If the abstract is vague, incomplete, or insufficient to decide safely, return MAYBE.
5. Be conservative: prefer MAYBE over incorrect EXCLUDE.
6. This is only title/abstract screening, not final inclusion.
7. Return valid JSON only.

ARTICLE:
Title: {title}
Abstract: {abstract}
Year: {year}
Source: {source}
DOI: {doi}
Conclusion: {conclusion}

Return exactly this JSON schema:
{{
  "decision": "INCLUDE | EXCLUDE | MAYBE",
  "criterion_triggered": ["criterion text 1", "criterion text 2"],
  "rationale": "short justification in English",
  "confidence": "HIGH | MEDIUM | LOW"
}}
""".strip()

## 6) Funções utilitárias


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\x00", " ").strip()

def extract_json(text: str):
    text = text.strip()

    # tenta parse direto
    try:
        return json.loads(text)
    except:
        pass

    # tenta capturar primeiro bloco JSON
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass

    return None

def normalize_decision(x: str):
    if not x:
        return "MAYBE"
    x = str(x).strip().upper()
    if x in {"INCLUDE", "EXCLUDE", "MAYBE"}:
        return x
    return "MAYBE"

## 7) Chamada da API


In [ ]:
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    retry=retry_if_exception_type((APITimeoutError, APIConnectionError, RateLimitError, APIError))
)
def classify_record(title, abstract, year="", source="", doi="", conclusion=""):
    prompt = build_prompt(title, abstract, year, source, doi, conclusion)

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "developer",
                "content": (
                    "You are a rigorous literature-screening assistant. "
                    "Return valid JSON only. Do not add markdown or extra commentary."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw = response.output_text
    parsed = extract_json(raw)

    if parsed is None:
        return {
            "decision": "MAYBE",
            "criterion_triggered": ["JSON_PARSE_ERROR"],
            "rationale": raw[:500],
            "confidence": "LOW",
            "raw": raw
        }

    return {
        "decision": normalize_decision(parsed.get("decision", "MAYBE")),
        "criterion_triggered": parsed.get("criterion_triggered", []),
        "rationale": parsed.get("rationale", ""),
        "confidence": parsed.get("confidence", "LOW"),
        "raw": raw
    }


## 8) Carregar dados


In [ ]:
df = pd.read_csv(INPUT_CSV)

# Normaliza nomes de colunas mínimas
lower_map = {c.lower().strip(): c for c in df.columns}

def pick_col(name_options, default=None):
    for name in name_options:
        if name in lower_map:
            return lower_map[name]
    return default

col_title = pick_col(["title"])
col_abstract = pick_col(["abstract"])
col_year = pick_col(["year", "publication year"])
col_source = pick_col(["source", "database", "origin"])
col_doi = pick_col(["doi"])
col_conclusion = pick_col(["conclusion", "conclusions"])

if col_title is None:
    raise ValueError("A planilha precisa ter uma coluna 'title'.")
if col_abstract is None:
    raise ValueError("A planilha precisa ter uma coluna 'abstract'.")

# Garante colunas auxiliares
if "record_id" not in df.columns:
    df.insert(0, "record_id", range(1, len(df) + 1))

extra_cols = [
    "llm_decision",
    "llm_criteria",
    "llm_rationale",
    "llm_confidence",
    "llm_raw",
    "model",
    "run_at",
    "prompt_version",
    "human_decision",
    "human_reason",
    "human_notes"
]
for c in extra_cols:
    if c not in df.columns:
        df[c] = ""

## 9) Retomar execução, se já existir saída


In [ ]:
out_path = Path(OUTPUT_CSV)
if out_path.exists():
    old = pd.read_csv(OUTPUT_CSV)
    if "record_id" in old.columns and "llm_decision" in old.columns:
        done_ids = set(old.loc[old["llm_decision"].fillna("") != "", "record_id"].tolist())
        print(f"Retomando execução. Registros já processados: {len(done_ids)}")
        # preserva o que já tinha
        old = old.set_index("record_id")
        df = df.set_index("record_id")
        for c in df.columns:
            if c in old.columns:
                df.loc[old.index.intersection(df.index), c] = old.loc[old.index.intersection(df.index), c]
        df = df.reset_index()
    else:
        done_ids = set()
else:
    done_ids = set()

## 10) Loop principal


In [ ]:
processed_since_save = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Screening"):
    record_id = row["record_id"]

    if record_id in done_ids:
        continue

    title = clean_text(row[col_title])
    abstract = clean_text(row[col_abstract])
    year = clean_text(row[col_year]) if col_year else ""
    source = clean_text(row[col_source]) if col_source else ""
    doi = clean_text(row[col_doi]) if col_doi else ""
    conclusion = clean_text(row[col_conclusion]) if col_conclusion else ""

    # Regra importante: não excluir automaticamente se não houver abstract
    if not abstract:
        decision = "MAYBE"
        criteria = ["NO_ABSTRACT"]
        rationale = "No abstract available for safe title/abstract screening."
        confidence = "LOW"
        raw = "NO_ABSTRACT"
    else:
        result = classify_record(
            title=title,
            abstract=abstract,
            year=year,
            source=source,
            doi=doi,
            conclusion=conclusion
        )
        decision = result["decision"]
        criteria = result["criterion_triggered"]
        rationale = result["rationale"]
        confidence = result["confidence"]
        raw = result["raw"]

    df.at[idx, "llm_decision"] = decision
    df.at[idx, "llm_criteria"] = "; ".join(criteria) if isinstance(criteria, list) else str(criteria)
    df.at[idx, "llm_rationale"] = rationale
    df.at[idx, "llm_confidence"] = confidence
    df.at[idx, "llm_raw"] = raw
    df.at[idx, "model"] = MODEL
    df.at[idx, "run_at"] = datetime.now(timezone.utc).isoformat()
    df.at[idx, "prompt_version"] = PROMPT_VERSION

    processed_since_save += 1

    if processed_since_save >= SAVE_EVERY_N:
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        processed_since_save = 0

    time.sleep(REQUEST_PAUSE_SECONDS)

# salva final
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Concluído. Arquivo salvo em: {OUTPUT_CSV}")

Screening:   0%|          | 0/76 [00:00<?, ?it/s]

Concluído. Arquivo salvo em: /content/drive/MyDrive/Colab Notebooks/TCC/screened_artigos.csv


## 11) Resumo rápido


In [ ]:
summary = df["llm_decision"].value_counts(dropna=False)
print("\nResumo das decisões sugeridas:")
print(summary)

print("\nPrimeiras linhas:")
display(df.head())


Resumo das decisões sugeridas:
llm_decision
EXCLUDE    49
INCLUDE    17
MAYBE      10
Name: count, dtype: int64

Primeiras linhas:


,record_id,title,abstract,doi,year,sources_merged,records_merged,llm_decision,llm_criteria,llm_rationale,llm_confidence,llm_raw,model,run_at,prompt_version,human_decision,human_reason,human_notes
0,1,13th Asian Symposium on Programming Languages ...,The proceedings contain 26 papers. The special...,NaN,2015,Scopus,1,EXCLUDE,Primary study published in a peer-reviewed ven...,"This record is for conference proceedings, not...",HIGH,"{\n ""decision"": ""EXCLUDE"",\n ""criterion_trig...",gpt-5.4,2026-03-29T20:18:43.187257+00:00,js_screening_v2,,,
1,2,20th European Conference on the Applications o...,The proceedings contain 15 papers. The special...,NaN,2017,Scopus,1,EXCLUDE,"Grey literature, such as thesis, dissertation,...","This record is for conference proceedings, not...",HIGH,"{\n ""decision"": ""EXCLUDE"",\n ""criterion_trig...",gpt-5.4,2026-03-29T20:18:46.948914+00:00,js_screening_v2,,,
2,3,26th International Conference on Fundamental A...,The proceedings contain 18 papers. The special...,NaN,2023,Scopus,1,EXCLUDE,"Secondary study, such as a systematic review, ...",This is a conference proceedings overview list...,HIGH,"{\n ""decision"": ""EXCLUDE"",\n ""criterion_trig...",gpt-5.4,2026-03-29T20:18:50.766712+00:00,js_screening_v2,,,
3,4,A Multivocal Mapping Study of MongoDB Smells,Code smells are symptoms of poor design or bad...,10.1109/SANER60148.2024.00086,2024,IEEE; Scopus,2,EXCLUDE,"Secondary study, such as a systematic review, ...",The paper is explicitly a multivocal mapping s...,HIGH,"{\n ""decision"": ""EXCLUDE"",\n ""criterion_trig...",gpt-5.4,2026-03-29T20:18:53.662526+00:00,js_screening_v2,,,
4,5,A large scale study on how developers discuss ...,"Context: In this paper, we investigate how dev...",10.1016/j.infsof.2020.106333,2020,Scopus,1,EXCLUDE,Main focus is not JavaScript.; Does not presen...,The study analyzes how developers discuss code...,HIGH,"{\n ""decision"": ""EXCLUDE"",\n ""criterion_trig...",gpt-5.4,2026-03-29T20:18:56.673529+00:00,js_screening_v2,,,
